# Grouping Data

Sometimes we want to select data based on groups and understand aggregated data on a group level. We have seen that even though Pandas allows us to iterate over every row in a dataframe, it is generally very slow to do so. Fortunately Pandas has a groupby() function to speed up such task. The idea behind the groupby() function is  that it takes some dataframe, splits it into chunks based on some key values, applies computation on those  chunks, then combines the results back together into another dataframe. In pandas this is referred to as the split-apply-combine pattern.

## Splitting

In [54]:
# Let's look at an example. First, we'll bring in our pandas and numpy libraries
import pandas as pd
import numpy as np

In [55]:
# Let's look at some US census data
df = pd.read_csv('census.csv')
# And exclude state level summarizations, which have sum level value of 40
df = df[df['SUMLEV']==50]
df.head()

,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861
5,50,3,6,1,9,Alabama,Blount County,57322,57322,57373,...,1.807375,-1.177622,-1.748766,-2.062535,-1.369970,1.859511,-0.848580,-1.402476,-1.577232,-0.884411


In [3]:
# In the first example for groupby() I want to use the census date. Let's get a list of the unique states,
# then we can iterate over all the states and for each state we reduce the data frame and calculate the
# average.

# Let's run such task for 3 times and time it. For this we'll use the cell magic function %%timeit

In [48]:
%%timeit -n 3

for state in df['STNAME'].unique():
    # We'll just calculate the average using numpy for this particular state
    avg = np.average(df.where(df['STNAME']==state).dropna()['CENSUS2010POP'])
    # And we'll print it to the screen
    print('Counties in state ' + state + 
          ' have an average population of ' + str(avg))

Counties in state Alabama have an average population of 71339.34328358209
Counties in state Alaska have an average population of 24490.724137931036
Counties in state Arizona have an average population of 426134.4666666667
Counties in state Arkansas have an average population of 38878.90666666667
Counties in state California have an average population of 642309.5862068966
Counties in state Colorado have an average population of 78581.1875
Counties in state Connecticut have an average population of 446762.125
Counties in state Delaware have an average population of 299311.3333333333
Counties in state District of Columbia have an average population of 601723.0
Counties in state Florida have an average population of 280616.5671641791
Counties in state Georgia have an average population of 60928.63522012578
Counties in state Hawaii have an average population of 272060.2
Counties in state Idaho have an average population of 35626.86363636364
Counties in state Illinois have an average populat

In [5]:
# If you scroll down to the bottom of that output you can see it takes a fair bit of time to finish.
# Now let's try another approach using groupby()

In [49]:
%%timeit -n 3
# For this method, we start by telling pandas we're interested in grouping by state name, this is the "split"
for group, frame in df.groupby('STNAME'):
    # You'll notice there are two values we set here. groupby() returns a tuple, where the first value is the
    # value of the key we were trying to group by, in this case a specific state name, and the second one is
    # projected dataframe that was found for that group
    
    # Now we include our logic in the "apply" step, which is to calculate an average of the census2010pop
    avg = np.average(frame['CENSUS2010POP'])
    # And print the results
    print('Counties in state ' + group + 
          ' have an average population of ' + str(avg))
# And we don't have to worry about the combine step in this case, because all of our data transformation is
# actually printing out results.

Counties in state Alabama have an average population of 71339.34328358209
Counties in state Alaska have an average population of 24490.724137931036
Counties in state Arizona have an average population of 426134.4666666667
Counties in state Arkansas have an average population of 38878.90666666667
Counties in state California have an average population of 642309.5862068966
Counties in state Colorado have an average population of 78581.1875
Counties in state Connecticut have an average population of 446762.125
Counties in state Delaware have an average population of 299311.3333333333
Counties in state District of Columbia have an average population of 601723.0
Counties in state Florida have an average population of 280616.5671641791
Counties in state Georgia have an average population of 60928.63522012578
Counties in state Hawaii have an average population of 272060.2
Counties in state Idaho have an average population of 35626.86363636364
Counties in state Illinois have an average populat

In [8]:
# Wow, what a huge difference in speed. An improve by roughly by two factors!

In [56]:
# Now, 99% of the time, you'll use group by on one or more columns. But you can also provide a function to
# group by and use that to segment your data.

# This is a bit of a fabricated example but lets say that you have a big batch job with lots of processing and
# you want to work on only a third or so of the states at a given time. We could create some function which
# returns a number between zero and two based on the first character of the state name. Then we can tell group
# by to use this function to split up our data frame. It's important to note that in order to do this you need
# to set the index of the data frame to be the column that you want to group by first.

# We'll create some new function called set_batch_number and if the first letter of the parameter is a capital
# M we'll return a 0. If it's a capital Q we'll return a 1 and otherwise we'll return a 2. Then we'll pass
# this function to the data frame

df = df.set_index('STNAME')

def set_batch_number(item):
    if item[0]<'M':
        return 0
    if item[0]<'Q':
        return 1
    return 2

# The dataframe is supposed to be grouped by according to the batch number And we will loop through each batch
# group
for group, frame in df.groupby(set_batch_number):
    print('There are ' + str(len(frame)) + ' records in group ' + str(group) + ' for processing.')

There are 1177 records in group 0 for processing.
There are 1134 records in group 1 for processing.
There are 831 records in group 2 for processing.


In [6]:
# Notice that this time I didn't pass in a column name to groupby(). Instead, I set the index of the dataframe
# to be STNAME, and if no column identifier is passed groupby() will automatically use the index.

In [79]:
# Let's take one more look at an example of how we might group data. In this example, I want to use a dataset
# of housing from airbnb. In this dataset there are two columns of interest, one is the cancellation_policy
# and the other is the review_scores_value.
df=pd.read_csv("listings.csv")
df.head()
# df.columns

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,231497,https://www.airbnb.com/rooms/231497,20230922043705,2023-09-22,city scrape,Rental unit in Rio de Janeiro · ★4.73 · 1 bedr...,"This is a big studio at the end of Copacabana,...",NaN,https://a0.muscache.com/pictures/3582382/ee8ac...,1207700,...,4.92,4.90,4.65,NaN,f,4,4,0,0,0.54
1,231516,https://www.airbnb.com/rooms/231516,20230922043705,2023-09-22,city scrape,Rental unit in Rio de Janeiro · ★4.71 · 1 bedr...,"Special location of the building, on Copacaban...",NaN,https://a0.muscache.com/pictures/3671683/d74b4...,1207700,...,4.86,4.93,4.38,NaN,f,4,4,0,0,0.20
2,236991,https://www.airbnb.com/rooms/236991,20230922043705,2023-09-23,city scrape,Rental unit in Rio de Janeiro · ★4.89 · 1 bedr...,"Aconchegante, amplo, básico, arejado, iluminad...","Copacabana, apelidada a princesinha do mar, fa...",https://a0.muscache.com/pictures/5725a59b-147d...,1241662,...,4.96,4.99,4.89,NaN,f,2,2,0,0,0.65
3,17878,https://www.airbnb.com/rooms/17878,20230922043705,2023-09-23,city scrape,Condo in Rio de Janeiro · ★4.70 · 2 bedrooms ·...,Please note that elevated rates applies for Ne...,This is the one of the bests spots in Rio. Bec...,https://a0.muscache.com/pictures/65320518/3069...,68997,...,4.91,4.77,4.67,NaN,f,1,1,0,0,1.87
4,25026,https://www.airbnb.com/rooms/25026,20230922043705,2023-09-22,city scrape,Rental unit in Rio de Janeiro · ★4.71 · 1 bedr...,"**Fully renovated in Dec 2022, new kitchen, n...",Copacabana is a lively neighborhood and the ap...,https://a0.muscache.com/pictures/a745aa21-b8dd...,102840,...,4.92,4.84,4.59,NaN,f,1,1,0,0,1.68


In [63]:
# So, how would I group by both of these columns? A first approach might be to promote them to a multiindex
# and just call groupby()
df=df.set_index(["source","review_scores_value"])

# When we have a multiindex we need to pass in the levels we are interested in grouping by
for group, frame in df.groupby(level=(0,1)):
    print(group)

('city scrape', 1.0)
('city scrape', 1.5)
('city scrape', 2.0)
('city scrape', 2.33)
('city scrape', 2.5)
('city scrape', 2.67)
('city scrape', 2.75)
('city scrape', 3.0)
('city scrape', 3.17)
('city scrape', 3.2)
('city scrape', 3.25)
('city scrape', 3.29)
('city scrape', 3.33)
('city scrape', 3.4)
('city scrape', 3.43)
('city scrape', 3.5)
('city scrape', 3.56)
('city scrape', 3.57)
('city scrape', 3.6)
('city scrape', 3.62)
('city scrape', 3.63)
('city scrape', 3.64)
('city scrape', 3.67)
('city scrape', 3.7)
('city scrape', 3.71)
('city scrape', 3.75)
('city scrape', 3.78)
('city scrape', 3.79)
('city scrape', 3.8)
('city scrape', 3.82)
('city scrape', 3.83)
('city scrape', 3.85)
('city scrape', 3.86)
('city scrape', 3.88)
('city scrape', 3.89)
('city scrape', 3.9)
('city scrape', 3.91)
('city scrape', 3.92)
('city scrape', 3.93)
('city scrape', 3.94)
('city scrape', 3.95)
('city scrape', 4.0)
('city scrape', 4.03)
('city scrape', 4.04)
('city scrape', 4.05)
('city scrape', 4.06)
(

In [65]:
# This seems to work ok. But what if we wanted to group by the source and review scores, but
# separate out all the 10's from those under ten? In this case, we could use a function to manage the
# groupings

def grouping_fun(item):
    # Check the "review_scores_value" portion of the index. item is in the format of
    # (source,review_scores_value
    if item[1] == 5.0:
        return (item[0],"5.0")
    else:
        return (item[0],"not 5.0")

for group, frame in df.groupby(by=grouping_fun):
    print(group)

('city scrape', '5.0')
('city scrape', 'not 5.0')
('previous scrape', '5.0')
('previous scrape', 'not 5.0')


In [66]:
df.head()

id                          listing_url  \
source      review_scores_value                                                
city scrape 4.65                 231497  https://www.airbnb.com/rooms/231497   
            4.38                 231516  https://www.airbnb.com/rooms/231516   
            4.89                 236991  https://www.airbnb.com/rooms/236991   
            4.67                  17878   https://www.airbnb.com/rooms/17878   
            4.59                  25026   https://www.airbnb.com/rooms/25026   

                                      scrape_id last_scraped  \
source      review_scores_value                                
city scrape 4.65                 20230922043705   2023-09-22   
            4.38                 20230922043705   2023-09-22   
            4.89                 20230922043705   2023-09-23   
            4.67                 20230922043705   2023-09-23   
            4.59                 20230922043705   2023-09-22   

                                                                              name  \
source      review_scores_value                                                      
city scrape 4.65                 Rental unit in Rio de Janeiro · ★4.73 · 1 bedr...   
            4.38                 Rental unit in Rio de Janeiro · ★4.71 · 1 bedr...   
            4.89                 Rental unit in Rio de Janeiro · ★4.89 · 1 bedr...   
            4.67                 Condo in Rio de Janeiro · ★4.70 · 2 bedrooms ·...   
            4.59                 Rental unit in Rio de Janeiro · ★4.71 · 1 bedr...   

                                                                       description  \
source      review_scores_value                                                      
city scrape 4.65                 This is a big studio at the end of Copacabana,...   
            4.38                 Special location of the building, on Copacaban...   
            4.89                 Aconchegante, amplo, básico, arejado, iluminad...   
            4.67                 Please note that elevated rates applies for Ne...   
            4.59                 **Fully renovated  in Dec 2022, new kitchen, n...   

                                                             neighborhood_overview  \
source      review_scores_value                                                      
city scrape 4.65                                                               NaN   
            4.38                                                               NaN   
            4.89                 Copacabana, apelidada a princesinha do mar, fa...   
            4.67                 This is the one of the bests spots in Rio. Bec...   
            4.59                 Copacabana is a lively neighborhood and the ap...   

                                                                       picture_url  \
source      review_scores_value                                                      
city scrape 4.65                 https://a0.muscache.com/pictures/3582382/ee8ac...   
            4.38                 https://a0.muscache.com/pictures/3671683/d74b4...   
            4.89                 https://a0.muscache.com/pictures/5725a59b-147d...   
            4.67                 https://a0.muscache.com/pictures/65320518/3069...   
            4.59                 https://a0.muscache.com/pictures/a745aa21-b8dd...   

                                 host_id  \
source      review_scores_value            
city scrape 4.65                 1207700   
            4.38                 1207700   
            4.89                 1241662   
            4.67                   68997   
            4.59                  102840   

                                                                  host_url  \
source      review_scores_value                                              
city scrape 4.65                 https://www.airbnb.com/users/show/1207700   
            4.38                 https://www.airbnb.com/users/show/12077

## Applying

To this point we have applied very simple processing to our data after splitting, really just outputting some print statements to demonstrate how the splitting works. The pandas developers have three broad categories of data processing to happen during the apply step, Aggregation of group data, Transformation of group data, and Filtration of group data

### Aggregation

In [67]:
# The most straight forward apply step is the aggregation of data, and uses the method agg() on the groupby
# object. Thus far we have only iterated through the groupby object, unpacking it into a label (the group
# name) and a dataframe. But with agg we can pass in a dictionary of the columns we are interested in
# aggregating along with the function we are looking to apply to aggregate.

# Let's reset the index for our airbnb data
df=df.reset_index()

# Now lets group by the cancellation policy and find the average review_scores_value by group
df.groupby("source").agg({"review_scores_value":np.average})

,review_scores_value
source,
city scrape,NaN
previous scrape,NaN


In [76]:
# Hrm. That didn't seem to work at all. Just a bunch of not a numbers. The issue is actually in the function
# that we sent to aggregate. np.average does not ignore nans! However, there is a function we can use for this
df.groupby("source").agg({"review_scores_value":np.nanmean})

/var/folders/tl/5s97jcvx67bb0w5spkp3pz380000gn/T/ipykernel_64934/1357064532.py:3: FutureWarning: The provided callable <function nanmean at 0x10822f060> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby("source").agg({"review_scores_value":np.nanmean})


,review_scores_value
source,
city scrape,4.700186
previous scrape,4.658926


In [73]:
# We can just extend this dictionary to aggregate by multiple functions or multiple columns.
df.groupby("source").agg({"review_scores_value":(np.nanmean,np.nanstd),
                                      "reviews_per_month":np.nanmean})

/var/folders/tl/5s97jcvx67bb0w5spkp3pz380000gn/T/ipykernel_64934/951477428.py:2: FutureWarning: The provided callable <function nanmean at 0x10822f060> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby("source").agg({"review_scores_value":(np.nanmean,np.nanstd),
/var/folders/tl/5s97jcvx67bb0w5spkp3pz380000gn/T/ipykernel_64934/951477428.py:2: FutureWarning: The provided callable <function nanstd at 0x10822fa60> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  df.groupby("source").agg({"review_scores_value":(np.nanmean,np.nanstd),
/var/folders/tl/5s97jcvx67bb0w5spkp3pz380000gn/T/ipykernel_64934/951477428.py:2: FutureWarning: The provided callable <function nanmean at 0x10822f060> is currently using SeriesGroupBy.mean. In a future vers

review_scores_value           reviews_per_month
                            nanmean    nanstd           nanmean
source                                                         
city scrape                4.700186  0.435636          1.026820
previous scrape            4.658926  0.564001          0.428315

In [75]:
# Take a moment to make sure you understand the previous cell, since it's somewhat complex. First we're doing
# a group by on the dataframe object by the column "source". This creates a new GroupBy object.
# Then we are invoking the agg() function on that object. The agg function is going to apply one or more
# functions we specify to the group dataframes and return a single row per dataframe/group. When we called
# this function we sent it two dictionary entries, each with the key indicating which column we wanted
# functions applied to. For the first column we actually supplied a tuple of two functions. Note that these
# are not function invocations, like np.nanmean(), or function names, like "nanmean" they are references to
# functions which will return single values. The groupby object will recognize the tuple and call each
# function in order on the same column. The results will be in a heirarchical index, but since they are
# columns they don't show as an index per se. Then we indicated another column and a single function we wanted
# to run.

Groupby is a powerful and commonly used tool for data cleaning and data analysis. Once you have grouped the data by some category you have a dataframe of just those values and you can conduct aggregated analysis on the segments that you are interested. The groupby() function follows a split-apply-combine approach - first the data is split into subgroups, then you can apply some transformation, filtering, or aggregation, then the results are combined automatically by pandas for us.